In [1]:
#necessary imports
import sklearn as sk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from array_temp.DataPreprocessing import make_sequence_datasets


from data_tools import query
from data_tools.collections import TimeSeries
from datetime import datetime, date, time, timezone
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import dill
import os
import pytz
from datetime import datetime, time, date

In [2]:
# query data from influx. looking at timestamps of 2024 FSGP: 14 - 18 overall, but for smaller data response lets try 14 -16

#each 5 seconds
utc_offset_h = 7
start_utc = time(0+utc_offset_h, 00, 00)  #querying i svancouver time, influxdb gives utc
stop_utc = time(16+utc_offset_h, 45, 00)
date_start = date(2024, 7, 16)
date_stop = date(2024, 7, 18)

vancouver = pytz.timezone("America/Vancouver")

start_local = vancouver.localize(datetime.combine(date_start, start_utc))
stop_local = vancouver.localize(datetime.combine(date_stop, stop_utc))

start_time = start_local.astimezone(pytz.utc)
stop_time = stop_local.astimezone(pytz.utc)

client = query.DBClient()
mech_brake_pressed: TimeSeries = client.query_time_series(start_time, stop_time, field="MechBrakePressed")
accel_position: TimeSeries =   client.query_time_series(start_time, stop_time, field="AcceleratorPosition")
speed_kph: TimeSeries = client.query_time_series(start_time, stop_time, "VehicleVelocity")



In [3]:

# save collected data

out_dir = os.path.join("../../array_temp", "data", "control_state_fsgp_2024")
os.makedirs(out_dir, exist_ok=True)

brake_path = os.path.join(out_dir, "brake_pressed.bin")
accel_path = os.path.join(out_dir, "acceleration.bin")
speed_path = os.path.join(out_dir, "speed_kph.bin")

filepaths = [brake_path, accel_path, speed_path]
datasets = [mech_brake_pressed, accel_position, speed_kph]

for filepath, data in zip(filepaths, datasets):
    with open(filepath, "wb") as f:
        dill.dump(data, f)


In [4]:
#loading data
import os
import dill
out_dir = os.path.join("../../array_temp", "data", "control_state_fsgp_2024")

brake_path = os.path.join(out_dir, "brake_pressed.bin")
accel_path = os.path.join(out_dir, "acceleration.bin")
speed_path = os.path.join(out_dir, "speed_kph.bin")

filepaths = [brake_path, accel_path, speed_path]

loaded_datasets = []

for filepath in filepaths:
    with open(filepath, "rb") as f:
        data = dill.load(f)
        loaded_datasets.append(data)

#unnpack
mech_brake_pressed, accel_position, speed_kph = loaded_datasets

In [5]:
# use sunbeam instead to save yourself a headache
from data_tools import *
import numpy as np
def make_df(source, name):
    dfs = []

    client = query.SunbeamClient()

    for event in ["FSGP_2024_Day_1", "FSGP_2024_Day_2", "FSGP_2024_Day_3"]:
        file = client.get_file(
            origin="production",
            event=event,
            source=source,
            name=name
        ).unwrap()

        dfs.append(
            pd.DataFrame(
                data=file.data,
                index=file.data.datetime_x_axis
            )
        )

    return pd.concat(dfs).sort_index()

In [6]:
pos = []
pos_df = make_df(source = "localization", name = "TrackIndex")

In [7]:
speed_df = make_df(source = "ingress", name = "VehicleVelocity")

In [8]:
# combine all dfs and resample, then feed to scaler.
# states = velocity, position
# control = mbrake pressed, accelerator position
def combine_dfs(telemetry_names, index_common, all_dfs):
        combined_df = pd.DataFrame(index=index_common)
        combined_df.dropna()

        for name, df in zip(telemetry_names, all_dfs):
            #df_interp = self.resample(df, index_common)
            combined_df[name] = df

        return combined_df


In [9]:
#preprocessing - convert everything to pandas dataframes.


df_mech_brake_pressed = pd.DataFrame(mech_brake_pressed)
df_accel_position = pd.DataFrame(accel_position)
#df_speed_kph = pd.DataFrame(speed_kph)



all_dfs = [df_mech_brake_pressed, df_accel_position]
combined_df = combine_dfs(["mech_brake_pressed", "accel_position"], df_mech_brake_pressed.index, all_dfs)


In [11]:

dfs = pd.concat([pos_df, speed_df], axis = 1)
#df = (["position", "speed"], pos_df.index, dfs)

In [21]:
dfs = dfs.reindex(combined_df.index)
#speed_df = speed_df.reindex(combined_df.index)

final_df = pd.concat([combined_df, dfs], axis=1)

In [12]:
dfs.head()

,TrackIndexSpreadsheet,0
2024-07-16 07:49:53.627000093,0.0,0.0
2024-07-16 07:49:53.827000618,0.0,0.0
2024-07-16 07:49:54.027000904,0.0,0.0
2024-07-16 07:49:54.227001429,0.0,0.0
2024-07-16 07:49:54.427001953,0.0,0.0


In [22]:
final_df.head()

,mech_brake_pressed,accel_position,TrackIndexSpreadsheet,0
0,0.0,0.0,NaN,NaN
1,0.0,0.0,NaN,NaN
2,0.0,0.0,NaN,NaN
3,0.0,0.0,NaN,NaN
4,0.0,0.0,NaN,NaN
